# M2 · E3 pilot pada data olahan 640

Ulangi inferensi dan evaluasi dua checkpoint E3 historis pada Kaggle. Keduanya dilatih pada 1.240 sumber olahan 640 dengan 620 langkah optimizer per model. Kontrol historis menangani citra utuh selama empat epoch. Perlakuan historis menangani 4.960 tile 384 selama satu epoch.

Notebook membandingkan kedua checkpoint pada 356 citra validasi utuh, menghitung COCO AP-small dan angka ambang tetap, lalu mengukur latensi CPU. E3 revisi native 1280 merupakan eksperimen terpisah dengan geometri tile dan data latih yang berbeda.

In [ ]:
import os
import subprocess
import sys
os.environ["YOLO_CONFIG_DIR"] = "/tmp/ciringoni-yolo"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.210", "pycocotools==2.0.11"], check=True)

In [ ]:
from hashlib import file_digest
from importlib.metadata import version
import json
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd

SOURCE_COMMIT = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
ARMS = {
    "E3_control_full_640": "a03c79c060dd3ceb3cc9e0523890d69477b5da57f8a1a255f5852e685a8d18bb",
    "E3_tile384_640": "0da9c0da8f514b36416f7a5029d5622fc2c6fbccc39747687de86a0181a3354f",
}

def sha256(path):
    with path.open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()

input_root = Path("/kaggle/input")
scratch = TemporaryDirectory(prefix="ciringoni-e3-pilot-")
project = Path(scratch.name) / "Data_Mining_Ciringoni_M1"
processed = project / "processed"
processed.mkdir(parents=True)
gt_sources = list(input_root.rglob("coco_val_gt.json"))
assert len(gt_sources) == 1
gt_source = gt_sources[0]
zip_paths = list(gt_source.parent.glob("processed_train_val.zip"))
if zip_paths:
    with ZipFile(zip_paths[0]) as archive:
        paths = [name for name in archive.namelist() if name.startswith("dataset/") and "test" not in Path(name).parts]
        assert all(".." not in Path(name).parts for name in paths)
        archive.extractall(processed, members=paths)
    processed_input = processed / "dataset"
else:
    direct = gt_source.parent / "dataset"
    processed_input = direct if direct.exists() else gt_source.parent / "processed_train_val/dataset"
assert not (processed_input / "images/test").exists()
for directory in ("images", "labels"):
    (processed / directory).symlink_to(processed_input / directory, target_is_directory=True)
val_yaml = processed / "data.yaml"
val_yaml.write_text(json.dumps({"path": str(processed), "train": "images/train", "val": "images/val", "names": ["infected"]}))
ground_truth_path = project / "hasil_m1/coco_val_gt.json"
ground_truth_path.parent.mkdir()
shutil.copyfile(gt_source, ground_truth_path)
ground_truth = json.loads(ground_truth_path.read_text())
assert (len(ground_truth["images"]), len(ground_truth["annotations"])) == (356, 2205)
assert {path.name for path in (processed / "images/val").glob("*.jpg")} == {row["file_name"] for row in ground_truth["images"]}

sources = ["hasil_m2/evaluasi_mosaic.py", "FP_Data_Mining_M1_Eksekusi_Lokal.ipynb"]
source_hashes = {}
for name in sources:
    target = project / name
    target.parent.mkdir(exist_ok=True)
    urlretrieve(f"https://raw.githubusercontent.com/Jordyee/ciringoni-data-mining/{SOURCE_COMMIT}/Data_Mining_Ciringoni_M1/{name}", target)
    source_hashes[name] = sha256(target)
historical_info = {}
for arm, expected in ARMS.items():
    file, = input_root.rglob(f"{arm}.pt")
    assert sha256(file) == expected
    historical_training, = input_root.rglob(f"{arm}_steps.json")
    steps = json.loads(historical_training.read_text())
    assert steps["minibatches"] == steps["optimizer_steps"] == 620
    destination = project / f"hasil_m2/eksperimen_e3_640_verified/runs/{arm}/weights/last.pt"
    destination.parent.mkdir(parents=True)
    destination.symlink_to(file)
    historical_info[arm] = {"checkpoint_sha256": expected, "historical_epochs": steps["epochs"], "historical_optimizer_steps": steps["optimizer_steps"]}
print("356 gambar validasi, 2205 kotak acuan. SHA-256 checkpoint:", ARMS)

In [ ]:
available = sorted(os.sched_getaffinity(0))
assert len(available) >= 4, available
affinity = available[:4]
os.sched_setaffinity(0, affinity)
evaluator = project / "hasil_m2/evaluasi_mosaic.py"
command = [sys.executable, str(evaluator), str(val_yaml), "--device", "cpu", "--runs", *ARMS]
subprocess.run(command, check=True)
subprocess.run([*command, "--cpu-benchmark"], check=True)

In [ ]:
output = Path("/kaggle/working/E3-pilot")
evaluation = project / "hasil_m2/evaluasi"
output.mkdir()
rows = []
for arm in ARMS:
    original = evaluation / arm
    pred = original / "coco_val_predictions.json"
    metrics = json.loads((original / "metrik.json").read_text())
    assert (metrics["gambar"], metrics["split"], metrics["cpu_torch_threads"]) == (356, "val", 4)
    assert 0 < metrics["cpu_median_ms"] <= metrics["cpu_p95_ms"]
    predictions = json.loads(pred.read_text())
    assert predictions and {row["image_id"] for row in predictions} <= set(range(1, 357))
    evaluated_files = (original / "coco_val_inputs.txt").read_text().splitlines()
    assert len(evaluated_files) == 356 and {Path(path).name for path in evaluated_files} == {row["file_name"] for row in ground_truth["images"]}
    folder = output / arm
    folder.mkdir()
    shutil.copyfile(pred, folder / pred.name)
    shutil.copyfile(original / "metrik.json", folder / "metrik.json")
    rows.append(metrics | historical_info[arm] | {"predictions": len(predictions), "predictions_sha256": sha256(pred)})
table = pd.DataFrame(rows)
table.to_csv(output / "hasil_e3_pilot.csv", index=False)
display(table[["id", "AP-small", "mAP50-95", "presisi_conf_0.05_iou_0.5", "recall_conf_0.05_iou_0.5", "cpu_median_ms", "cpu_p95_ms"]])
proof = {
    "experiment": "E3 historical processed 640 pilot",
    "activity": ["Kaggle CPU inference", "COCO validation", "Kaggle CPU latency"],
    "training_performed_in_this_run": False,
    "source_commit": SOURCE_COMMIT, "source_sha256": source_hashes,
    "historical_training": historical_info, "ground_truth_sha256": sha256(gt_source),
    "images": 356, "gt_boxes": 2205, "cpu_logical_affinity": affinity,
    "python": sys.version.split()[0],
    "cpu": next(line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name")),
    "versions": {p: version(p) for p in ("ultralytics", "pycocotools", "torch", "torchvision")},
    "outputs_sha256": {str(path.relative_to(output)): sha256(path) for path in sorted(output.rglob("*")) if path.is_file()},
}
(output / "run_e3_pilot.json").write_text(json.dumps(proof, indent=2) + "\n")
print("Bukti E3 pilot tersimpan pada", output)